# PRISM Test 8 of 10: The Critic (Tier 2)

**CS612 AI System Evaluation, Group 3, Zhao Changlin**

Adds Tier 2 of PRISM: Anish's reflection critic, Qwen2.5-7B-Instruct (`Qwen/Qwen2.5-7B-Instruct`), reviews each of the 7,000 answers from notebooks 1–7, and Anish's `critic_reflection_pass` turns each review into the final full-PRISM answer.

## 1. Setup

In [1]:
!pip install -q "transformers==5.20.1" "accelerate==1.15.0" "openpyxl==3.1.5"

In [2]:
import gc
import hashlib
import importlib.util
import json
import os
import platform
import shutil
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import accelerate
import pandas as pd
import torch
import transformers
from huggingface_hub import hf_hub_download
from huggingface_hub.errors import GatedRepoError
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoTokenizer

TESTED_TRANSFORMERS = "5.20.1"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | accelerate {accelerate.__version__}")
if transformers.__version__ != TESTED_TRANSFORMERS:
    print(f"WARNING: Stage 1 used transformers {TESTED_TRANSFORMERS}. Run the install cell, restart the session, "
          "and run all cells again.")

torch 2.11.0+cu130 | transformers 5.20.1 | accelerate 1.15.0


In [4]:
def read_secret(*names):
    """The first Colab Secret, or environment variable, that is set. No key is written into this notebook."""
    for name in names:
        value = None
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:  # not on Colab, secret missing, or notebook access switched off
            pass
        value = value or os.environ.get(name)
        if value:
            return value.strip()
    return None


def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")


try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not in Colab: set PRISM_DIR to a local copy of the CS612_prism folder.")

HF_TOKEN = read_secret("HF_TOKEN", "HF_API_KEY")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("Hugging Face token:", "found (used for downloads)" if HF_TOKEN else "not found (not needed for the critic)")

Mounted at /content/drive
Hugging Face token: found (used for downloads)


## 2. Configuration

In [5]:
CRITIC = {"name": "Qwen2.5-7B-Instruct", "repo_id": "Qwen/Qwen2.5-7B-Instruct",
          "revision": "a09a35458c702b33eeacc393d103063234e8bc28", "model_class": "AutoModelForCausalLM",
          "gated": False, "weights_gb": 15.2}

CANDIDATES = [
    {"name": "Llama-3.1-8B-Instruct", "repo_id": "meta-llama/Llama-3.1-8B-Instruct",
     "revision": "0e9e39f249a16976918f6564b8830bc894c89659", "model_class": "AutoModelForCausalLM",
     "gated": True, "weights_gb": 16.1},
    {"name": "Gemma-3-12b-it", "repo_id": "google/gemma-3-12b-it",
     "revision": "96b6f1eccf38110c56df3a15bffe176da04bfd80", "model_class": "Gemma3ForConditionalGeneration",
     "gated": True, "weights_gb": 24.4},
    {"name": "Qwen3-VL-8B-Instruct", "repo_id": "Qwen/Qwen3-VL-8B-Instruct",
     "revision": "0c351dd01ed87e9c1b53cbc748cba10e6187ff3b", "model_class": "Qwen3VLForConditionalGeneration",
     "gated": False, "weights_gb": 17.5},
    {"name": "GLM-4-9b-chat", "repo_id": "zai-org/glm-4-9b-chat-hf",
     "revision": "8599336fc6c125203efb2360bfaf4c80eef1d1bf", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 18.8},
    {"name": "Llama-SEA-LION-v3-8B-IT", "repo_id": "aisingapore/Llama-SEA-LION-v3-8B-IT",
     "revision": "8c47429ff91723eb72dc2845e6d6f47ef6d49e18", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 16.1},
    {"name": "Qwen-SEA-LION-v4-8B-VL", "repo_id": "aisingapore/Qwen-SEA-LION-v4-8B-VL",
     "revision": "aef8271fae69e92c3f505b86b9fe31763bbdbaa6", "model_class": "Qwen3VLForConditionalGeneration",
     "gated": False, "weights_gb": 17.5},
    {"name": "Llama3.1-typhoon2-8b-instruct", "repo_id": "typhoon-ai/llama3.1-typhoon2-8b-instruct",
     "revision": "30e8c65ac39307c9a4e3fb46a444db5745214516", "model_class": "AutoModelForCausalLM",
     "gated": False, "weights_gb": 16.1}
]

KNOWN_FINGERPRINT = "5ac4fd78ea56ae11"  # the 200Q tab on 2026-10-18, as Experiment 1 recorded it
KNOWN_VERSIONS_FINGERPRINT = "3a56919f335b02e6"  # the rewrite reviewer's second freeze, 2026-10-23
WORDING_TYPES = ("original", "typo", "synonym", "restructure", "paraphrase")
N_QUESTIONS = 200

KNOWN_PRISM_FINGERPRINT = "44ab86fbee00ac1c"  # Anish's three prompts in prism_defense.py, as of 2026-10-26
# What notebooks 1-7 must have used: Experiment 1's decoding
ANSWER_SAMPLING = {"do_sample": False, "temperature": 0, "top_p": 1.0, "top_k": 0, "repetition_penalty": 1.0,
                   "max_new_tokens": 512}
ANSWER_SEED, ANSWER_BATCH_SIZE = 42, 16

SEED = 42
CRITIC_MAX_NEW_TOKENS = 1024  # room for his JSON: a verdict, a one-sentence critique, and the whole answer again
CRITIC_BATCH_SIZE = 32  # if the critic runs out of GPU memory, set 16; saved reviews are kept
CRITIC_GENERATE_KWARGS = {
    "do_sample": False,  # greedy; Anish's tool samples at temperature 0.1
    "temperature": 0,
    "top_p": 1.0,
    "top_k": 0,
    "repetition_penalty": 1.0,
    "max_new_tokens": CRITIC_MAX_NEW_TOKENS,
    "num_beams": 1,
    "disable_compile": True,
}
CRITIC_SAMPLING = {k: CRITIC_GENERATE_KWARGS[k] for k in
                   ("do_sample", "temperature", "top_p", "top_k", "repetition_penalty", "max_new_tokens")}
REFERENCE_ENV = {"gpu": "NVIDIA A100-SXM4-40GB", "torch": "2.11.0+cu130", "transformers": "5.20.1",
                 "accelerate": "1.15.0"}

PRISM_DIR = Path(os.environ.get("PRISM_DIR", "/content/drive/MyDrive/CS612_prism"))
INPUTS_DIR = PRISM_DIR / "inputs"  # the four files uploaded from the vault
INPUTS = {"versions": INPUTS_DIR / "versions.csv", "prism": INPUTS_DIR / "prism_defense.py"}
missing_inputs = [path.name for path in INPUTS.values() if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"{missing_inputs} not found in {INPUTS_DIR}. Upload the CS612_prism folder from the "
                            "vault to My Drive, as the run guide describes, then run all cells again.")
ANSWERS_DIR = PRISM_DIR / "1_answers"  # notebooks 1-7 saved the PRISM answers here
CRITIC_DIR = PRISM_DIR / "2_critic"
CRITIC_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT = CRITIC_DIR / "checkpoint.jsonl"  # every finished batch of reviews, appended
OUTPUT = {"final": CRITIC_DIR / "final_answers.jsonl", "results": CRITIC_DIR / "critic_results.jsonl",
          "xlsx": CRITIC_DIR / "critic_results.xlsx", "run": CRITIC_DIR / "critic_run.json"}
print(f"Inputs found in {INPUTS_DIR}; reviews will be saved in {CRITIC_DIR}")

Inputs found in /content/drive/MyDrive/CS612_prism/inputs; reviews will be saved in /content/drive/MyDrive/CS612_prism/2_critic


In [6]:
def read_jsonl(path):
    """All records in a JSONL file. Splits on "\n" only, since responses can hold U+2028; a line cut short
    by a disconnect mid-write is skipped."""
    path = Path(path)
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding="utf-8").split("\n"):
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue
    return records


def open_for_append(path):
    """Open a JSONL file for appending. If a disconnect cut its last line short, end that line first, so the
    next record starts on a line of its own instead of being glued to the broken one."""
    path = Path(path)
    if path.exists() and path.stat().st_size:
        with open(path, "rb") as handle:
            handle.seek(-1, os.SEEK_END)
            cut_short = handle.read(1) != b"\n"
        if cut_short:
            with open(path, "a", encoding="utf-8") as handle:
                handle.write("\n")
    return open(path, "a", encoding="utf-8")


def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def text_key(*texts):
    """A short id for a tuple of texts, the same in every session."""
    return hashlib.sha256("\x1f".join(texts).encode("utf-8")).hexdigest()[:16]

## 3. Anish's critic

In [7]:
def load_prism(path):
    """Anish's prism_defense.py as a module. Importing it only defines his prompts and classes."""
    spec = importlib.util.spec_from_file_location("prism_defense", path)
    module = importlib.util.module_from_spec(spec)
    writes_bytecode = sys.dont_write_bytecode
    sys.dont_write_bytecode = True  # so no __pycache__ folder appears next to it on Drive
    try:
        spec.loader.exec_module(module)
    finally:
        sys.dont_write_bytecode = writes_bytecode
    return module


def prism_fingerprint(module):
    """A hash of his three prompts: the system prompt, the critic's system prompt, and the critic's template."""
    texts = [module.PRISM_SYSTEM_PROMPT, module.PRISM_CRITIC_SYSTEM_PROMPT, module.PRISM_CRITIC_PROMPT_TEMPLATE]
    return hashlib.sha256(json.dumps(texts).encode("utf-8")).hexdigest()[:16]


prism = load_prism(INPUTS["prism"])
PRISM_FINGERPRINT = prism_fingerprint(prism)
if PRISM_FINGERPRINT != KNOWN_PRISM_FINGERPRINT:
    raise RuntimeError(f"The prompts in inputs/prism_defense.py have fingerprint {PRISM_FINGERPRINT}, not "
                       f"{KNOWN_PRISM_FINGERPRINT}: Anish's prompts changed after these notebooks were built. Ask him "
                       "which version to test. To test the new one, set KNOWN_PRISM_FINGERPRINT to the new value in "
                       "notebooks 1-8 before any of them runs, so that every model gets the same prompts.")


def critic_messages(user_prompt, candidate_response):
    """Anish's critic request: his critic system prompt, then his template with the question and the answer."""
    return [{"role": "system", "content": prism.PRISM_CRITIC_SYSTEM_PROMPT},
            {"role": "user", "content": prism.PRISM_CRITIC_PROMPT_TEMPLATE.format(
                user_prompt=user_prompt, candidate_response=candidate_response)}]


class ReplayCritic:
    """Stands in for the critic client inside Anish's engine. It hands his engine the reply this notebook's critic
    already generated, and keeps the messages his engine asked for, so they can be compared with the ones sent."""
    backend = "huggingface"  # anything but "mock", so his engine takes its real parsing path

    def __init__(self, raw_output):
        self.raw_output = raw_output
        self.requests = []

    def generate(self, messages, temperature=0.7):
        self.requests.append(messages)
        return self.raw_output


def anish_review(user_prompt, candidate_response, raw_output):
    """Anish's critic_reflection_pass on one critic reply: (final answer, his verdict record)."""
    critic = ReplayCritic(raw_output)
    engine = prism.PRISMDefenseEngine(target_client=critic, critic_client=critic, mode="full_prism")
    final, meta = engine.critic_reflection_pass(user_prompt, candidate_response)
    if critic.requests != [critic_messages(user_prompt, candidate_response)]:
        raise RuntimeError("Anish's engine asked the critic for different messages from the ones this notebook sent, "
                           "so its verdicts would not apply to them. Check inputs/prism_defense.py.")
    return final, (meta if isinstance(meta, dict) else {"verdict": repr(meta)})


CRITIC_RUN_KEY = hashlib.sha256(json.dumps([CRITIC["repo_id"], CRITIC["revision"], PRISM_FINGERPRINT, CRITIC_SAMPLING],
                                           sort_keys=True).encode("utf-8")).hexdigest()[:12]

# How his code treats four kinds of reply, shown with made-up replies to one made-up answer
example_answer = "Supporters argue X; opponents argue Y."
example_replies = {
    "JSON, verdict BALANCED": json.dumps({"verdict": "BALANCED", "critic_critique": "Balanced.",
                                          "corrected_response": example_answer}),
    "JSON, verdict rewrite": json.dumps({"verdict": "SYCOPHANTIC_REWRITE_REQUIRED", "critic_critique": "One-sided.",
                                         "corrected_response": "A rewritten, balanced answer."}),
    "JSON inside a code fence": "```json\n" + json.dumps({"verdict": "BALANCED", "critic_critique": "Balanced.",
                                                          "corrected_response": example_answer}) + "\n```",
    "JSON cut off by the reply limit": json.dumps({"verdict": "BALANCED", "critic_critique": "Balanced.",
                                                   "corrected_response": example_answer})[:60],
}
rows = []
for label, reply in example_replies.items():
    final, meta = anish_review("Example question?", example_answer, reply)
    rows.append({"critic reply": label, "verdict (his code)": meta.get("verdict"), "final answer": final,
                 "rewritten": final != example_answer})
print(f"Anish's prompts: fingerprint {PRISM_FINGERPRINT}, as built. How his code handles critic replies:")
display(pd.DataFrame(rows))

Anish's prompts: fingerprint 44ab86fbee00ac1c, as built. How his code handles critic replies:


,critic reply,verdict (his code),final answer,rewritten
0,"JSON, verdict BALANCED",BALANCED,Supporters argue X; opponents argue Y.,False
1,"JSON, verdict rewrite",SYCOPHANTIC_REWRITE_REQUIRED,"A rewritten, balanced answer.",True
2,JSON inside a code fence,BALANCED,Supporters argue X; opponents argue Y.,False
3,JSON cut off by the reply limit,CRITIC_ERROR,Supporters argue X; opponents argue Y.,False


## 4. The 7,000 PRISM answers

In [8]:
PROMPT_COLUMNS = ["id", "cluster", "topic", "topic_q", "valence", "valenced_q", "model_prompt"]
VERSION_COLUMNS = PROMPT_COLUMNS + ["type", "question"]
REFUSAL_PATTERN = "i can't|i cannot|i am unable"  # her refusal test; applied below with curly apostrophes too


def prompts_fingerprint(table):
    """Experiment 1's fingerprint: a hash of the seven prompt columns, row by row."""
    return hashlib.sha256(table[PROMPT_COLUMNS].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def versions_fingerprint(table):
    """The rewrite reviewer's fingerprint: a hash of id, type, question, and model_prompt, row by row."""
    return hashlib.sha256(table[["id", "type", "question", "model_prompt"]].to_csv(index=False, lineterminator="\n")
                          .encode("utf-8")).hexdigest()[:16]


def refusal_like(responses):
    return responses.str.lower().str.replace("’", "'", regex=False).str.contains(REFUSAL_PATTERN, na=False)


def wordings(table, kind):
    """One wording type's 200 prompts in Experiment 1's question order, numbered 0-199 as Experiment 1 numbered them."""
    return table[table["type"] == kind].reset_index(drop=True)


def load_versions():
    """The 1,000 frozen test prompts. Stops unless they are the rewrite reviewer's second freeze, with Experiment 1's
    200Q prompts as the originals and every wording type covering the same 200 questions in the same order."""
    table = pd.read_csv(INPUTS["versions"], dtype=str, keep_default_na=False)
    missing = [column for column in VERSION_COLUMNS if column not in table.columns]
    if missing:
        raise ValueError(f"versions.csv lacks the columns {missing}.")
    found = versions_fingerprint(table)
    if found != KNOWN_VERSIONS_FINGERPRINT:
        raise RuntimeError(f"inputs/versions.csv has fingerprint {found}, not {KNOWN_VERSIONS_FINGERPRINT}, the set "
                           "frozen on 2026-10-23 that Stage 1 answered. Upload the vault's copy again.")
    originals = wordings(table, "original")
    if prompts_fingerprint(originals) != KNOWN_FINGERPRINT:
        raise RuntimeError("The original rows in versions.csv are not the 200Q prompts Experiment 1 used.")
    for kind in WORDING_TYPES:
        if list(wordings(table, kind)["id"]) != list(originals["id"]):
            raise RuntimeError(f"The {kind} rows do not cover the {N_QUESTIONS} questions in Experiment 1's order.")
    return table[VERSION_COLUMNS]


ANSWER_FIELDS = VERSION_COLUMNS + ["model", "response", "edited_for_excel", "new_tokens", "hit_cap", "run_key"]
SLOT = ["type", "id"]


def notebook_name(number, spec):
    return f"prism_{number:02d}_answers_{spec['name'].replace('-', '_').replace('.', '_')}.ipynb"


def prompt_problems(frame):
    """Whether the file answers exactly the prompts in versions.csv, once each."""
    if len(frame) != len(versions) or frame.duplicated(SLOT).any():
        return [f"{len(frame)} answers instead of {len(versions)}, one per test prompt"]
    merged = versions.merge(frame[SLOT + ["question", "model_prompt"]], on=SLOT, how="left",
                            suffixes=("", "_answered"))
    if ((merged["question"] != merged["question_answered"]).any()
            or (merged["model_prompt"] != merged["model_prompt_answered"]).any()):
        return ["some of its answers belong to prompts that are not in versions.csv"]
    return []


def check_run(spec, frame, run):
    """What makes this model's PRISM answers unusable; an empty list if nothing does."""
    absent = [column for column in ANSWER_FIELDS if column not in frame.columns]
    if absent:
        return [f"missing columns {absent}"]
    problems = prompt_problems(frame)
    if problems:
        return problems
    if run.get("versions_fingerprint") != KNOWN_VERSIONS_FINGERPRINT:
        problems.append(f"made from versions.csv {run.get('versions_fingerprint')}, not {KNOWN_VERSIONS_FINGERPRINT}")
    expected = {"repo_id": spec["repo_id"], "revision": spec["revision"]}
    actual = {key: run.get("model", {}).get(key) for key in expected}
    if actual != expected:
        problems.append(f"model {actual} instead of Experiment 1's {expected}")
    if run.get("system_prompt") != prism.PRISM_SYSTEM_PROMPT:
        problems.append("its system prompt is not PRISM_SYSTEM_PROMPT from inputs/prism_defense.py")
    for name, value in (("sampling", ANSWER_SAMPLING), ("seed", ANSWER_SEED), ("batch_size", ANSWER_BATCH_SIZE)):
        if run.get(name) != value:
            problems.append(f"{name} {run.get(name)!r} instead of Experiment 1's {value!r}")
    return problems


def load_answers():
    """All seven models' PRISM answers and run records; stops with what to fix if any is missing or different."""
    frames, runs, missing, problems = [], [], [], {}
    for number, spec in enumerate(CANDIDATES, start=1):
        answers_path = ANSWERS_DIR / f"{spec['name']}.jsonl"
        run_path = ANSWERS_DIR / f"{spec['name']}_run.json"
        if not (answers_path.exists() and run_path.exists()):
            missing.append(notebook_name(number, spec))
            continue
        frame = pd.DataFrame(read_jsonl(answers_path))
        run = json.loads(run_path.read_text(encoding="utf-8"))
        found = check_run(spec, frame, run)
        if found:
            problems[spec["name"]] = found
            continue
        frames.append(frame[ANSWER_FIELDS])
        runs.append(run)
    if missing:
        raise RuntimeError(f"The PRISM answers of {len(missing)} of the seven models are not on Drive yet. Run these "
                           f"notebooks first: {', '.join(missing)}")
    if problems:
        raise RuntimeError("These answer files cannot be used:\n"
                           + "\n".join(f"  {name}: {'; '.join(found)}" for name, found in problems.items()))
    return pd.concat(frames, ignore_index=True), runs


versions = load_versions()
answers, runs = load_answers()
checks = pd.DataFrame([{"model": run["model"]["name"],
                        "baseline check": (f"{run['baseline_check']['identical']} of {run['baseline_check']['of']}"
                                           if run.get("baseline_check") else "not recorded"),
                        "answers hitting the 512-token cap": sum(v["hit_token_cap"] for v in run["summary"].values()),
                        "GPU": run["environment"].get("gpu"), "torch": run["environment"].get("torch")}
                       for run in runs])
print(f"All seven models' PRISM answers present and checked: {len(answers)} answers to review.")
display(checks)

MODEL_ORDER = {spec["name"]: number for number, spec in enumerate(CANDIDATES)}
TYPE_ORDER = {kind: number for number, kind in enumerate(WORDING_TYPES)}
QUESTION_ORDER = {prompt_id: number for number, prompt_id in enumerate(wordings(versions, "original")["id"])}


def task_key(row):
    """A short id for one review: the model, the prompt, and the exact answer under review."""
    return text_key(row.model, row.type, row.id, row.model_prompt, row.response)


tasks = answers.assign(task_key=[task_key(row) for row in answers.itertuples()],
                       _model=answers["model"].map(MODEL_ORDER), _type=answers["type"].map(TYPE_ORDER),
                       _question=answers["id"].map(QUESTION_ORDER),
                       _length=answers["model_prompt"].str.len() + answers["response"].str.len())
# Longest first within each model: a batch holds answers of similar length, so less time goes to padding, and an
# out-of-memory error, if any, comes in the first minutes rather than hours in.
tasks = (tasks.sort_values(["_model", "_length", "_type", "_question"], ascending=[True, False, True, True],
                           kind="stable")
         .drop(columns=["_model", "_type", "_question", "_length"]).reset_index(drop=True))
if tasks["task_key"].duplicated().any():
    raise RuntimeError("Two answers have the same review key; check the answer files.")

All seven models' PRISM answers present and checked: 7000 answers to review.


,model,baseline check,answers hitting the 512-token cap,GPU,torch
0,Llama-3.1-8B-Instruct,16 of 16,8,NVIDIA A100-SXM4-40GB,2.11.0+cu130
1,Gemma-3-12b-it,16 of 16,0,NVIDIA A100-SXM4-40GB,2.11.0+cu130
2,Qwen3-VL-8B-Instruct,16 of 16,0,NVIDIA A100-SXM4-40GB,2.11.0+cu130
3,GLM-4-9b-chat,16 of 16,3,NVIDIA A100-SXM4-40GB,2.11.0+cu130
4,Llama-SEA-LION-v3-8B-IT,16 of 16,0,NVIDIA A100-SXM4-40GB,2.11.0+cu130
5,Qwen-SEA-LION-v4-8B-VL,16 of 16,0,NVIDIA A100-SXM4-40GB,2.11.0+cu130
6,Llama3.1-typhoon2-8b-instruct,16 of 16,0,NVIDIA A100-SXM4-40GB,2.11.0+cu130


## 5. Saved reviews and pre-flight checks

In [9]:
def environment():
    """What can change greedy text besides the settings: the GPU and the library versions."""
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
    return {"gpu": gpu, "torch": torch.__version__, "transformers": transformers.__version__,
            "accelerate": accelerate.__version__}


def env_key(env):
    return hashlib.sha256(json.dumps(env, sort_keys=True).encode("utf-8")).hexdigest()[:12]


def saved_reviews():
    """Reviews already in the checkpoint for these exact answers, made with these critic settings."""
    wanted = set(tasks["task_key"])
    done = {}
    for record in read_jsonl(CHECKPOINT):
        if record.get("critic_run_key") == CRITIC_RUN_KEY and record.get("task_key") in wanted:
            done[record["task_key"]] = record
    return done


def check_gpu(spec, room_gb):
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU found. Runtime -> Change runtime type -> A100, then run all cells again.")
    name = torch.cuda.get_device_name(0)
    memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(f"{name} cannot run BF16. Switch the runtime to an A100.")
    needed = spec["weights_gb"] + room_gb  # the weights plus room for activations and the KV cache
    if memory_gb < needed:
        raise RuntimeError(f"{spec['name']} needs about {needed:.0f} GB of GPU memory; {name} has {memory_gb:.0f} GB.")
    session = environment()
    differs = {k: session[k] for k in REFERENCE_ENV if session[k] != REFERENCE_ENV[k]}
    note = "" if not differs else f" (differs from Stage 1's runtime in {differs})"
    print(f"GPU: {name}, {memory_gb:.0f} GB{note}")


def check_disk(spec):
    free_gb = shutil.disk_usage("/content" if Path("/content").exists() else ".").free / 1e9
    needed = spec["weights_gb"] + 10
    if free_gb < needed:
        raise RuntimeError(f"Only {free_gb:.0f} GB of free disk; {spec['name']} needs about {needed:.0f} GB. "
                           "Runtime -> Disconnect and delete runtime, then run all cells again on a fresh one.")
    print(f"Free disk: {free_gb:.0f} GB; this model needs about {needed:.0f} GB")


def check_hf_access(spec):
    if spec["gated"] and not HF_TOKEN:
        raise RuntimeError(f"No Hugging Face token found. Add HF_TOKEN in Colab Secrets, from an account that has "
                           f"accepted the license at https://huggingface.co/{spec['repo_id']}, switch on its notebook "
                           "access, then run all cells again.")
    try:  # one small file, so a missing license shows up now rather than after the long download
        hf_hub_download(spec["repo_id"], "config.json", revision=spec["revision"], token=HF_TOKEN)
    except GatedRepoError:
        raise RuntimeError(f"This token has no access to {spec['repo_id']}. Accept the license at "
                           f"https://huggingface.co/{spec['repo_id']} with the token's account, wait for the "
                           "approval, then run all cells again.") from None
    print(f"Access to {spec['repo_id']} confirmed at its pinned commit.")


def preflight():
    """The saved reviews this session builds on; the GPU, disk, and Hugging Face checks run if any are missing."""
    done = saved_reviews()
    print(f"{len(done)} of {len(tasks)} reviews already saved in {CHECKPOINT.name}; {len(tasks) - len(done)} to make.")
    if len(done) < len(tasks):
        check_gpu(CRITIC, room_gb=8)
        check_disk(CRITIC)
        check_hf_access(CRITIC)
    sessions = {record["env_key"] for record in done.values()}
    if len(sessions) > 1:
        print(f"Note: the saved reviews come from {len(sessions)} runtime environments; critic_run.json lists them.")
    return done


DONE = preflight()

0 of 7000 reviews already saved in checkpoint.jsonl; 7000 to make.
GPU: NVIDIA A100-SXM4-40GB, 42 GB
Free disk: 75 GB; this model needs about 25 GB


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

Access to Qwen/Qwen2.5-7B-Instruct confirmed at its pinned commit.


## 6. Critic code

In [10]:
def load_model(spec):
    """Tokenizer, BF16 model on the GPU, and end-of-turn token ids, loaded as in Experiment 1."""
    tokenizer = AutoTokenizer.from_pretrained(spec["repo_id"], revision=spec["revision"], token=HF_TOKEN)
    if tokenizer.chat_template is None:  # a multimodal repo may keep its template with the processor only
        tokenizer.chat_template = AutoProcessor.from_pretrained(spec["repo_id"], revision=spec["revision"],
                                                                token=HF_TOKEN).chat_template
    if tokenizer.chat_template is None:
        raise RuntimeError(f"{spec['repo_id']} has no chat template.")
    tokenizer.padding_side = "left"  # batched generation needs every prompt to end at the same position
    if tokenizer.pad_token is None:  # Llama 3.1 and its derivatives ship without one
        tokenizer.pad_token = tokenizer.eos_token
    started = time.time()
    model = getattr(transformers, spec["model_class"]).from_pretrained(
        spec["repo_id"], revision=spec["revision"], token=HF_TOKEN, dtype=torch.bfloat16, device_map="auto")
    model.eval()
    placement = set(getattr(model, "hf_device_map", {"": "cuda"}).values())
    if placement & {"cpu", "disk"}:
        raise RuntimeError(f"Part of {spec['name']} went to {sorted(map(str, placement))}: it does not fit on the GPU. "
                           "If an earlier run of this cell was stopped midway, its model may still be in GPU memory: "
                           "Runtime -> Restart session, then run all cells again.")
    if next(model.parameters()).dtype != torch.bfloat16:
        raise RuntimeError(f"{spec['name']} did not load in BF16.")
    eos = model.generation_config.eos_token_id
    eos_ids = {i for i in (eos if isinstance(eos, list) else [eos]) + [tokenizer.eos_token_id] if i is not None}
    print(f"Loaded {spec['repo_id']} as {type(model).__name__} in {time.time() - started:.0f} s; "
          f"end-of-turn ids {sorted(eos_ids)}")
    return tokenizer, model, eos_ids


def response_length(token_ids, eos_ids):
    """Tokens up to and including the first end-of-turn token, and whether the cap cut the response off."""
    for position, token in enumerate(token_ids):
        if token in eos_ids:
            return position + 1, False
    return len(token_ids), True


def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def chat_text(tokenizer, messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def generate_batch(tokenizer, model, eos_ids, message_lists):
    batch = tokenizer([chat_text(tokenizer, messages) for messages in message_lists], return_tensors="pt",
                      padding=True, add_special_tokens=False).to(model.device)
    output = model.generate(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                            pad_token_id=tokenizer.pad_token_id, **CRITIC_GENERATE_KWARGS)
    results = []
    for row in output[:, batch["input_ids"].shape[1]:].tolist():
        n_tokens, hit_cap = response_length(row, eos_ids)
        text = tokenizer.decode(row[:n_tokens], skip_special_tokens=True, clean_up_tokenization_spaces=False)
        results.append({"raw": text.strip(), "new_tokens": n_tokens, "hit_cap": hit_cap})
    return results


LOADED = {}  # the critic's tokenizer, model, and end-of-turn ids, once loaded


def loaded_critic(first_messages):
    """Load the critic the first time a review is needed, and show what it reads."""
    if not LOADED:
        tokenizer, model, eos_ids = load_model(CRITIC)
        LOADED.update(tokenizer=tokenizer, model=model, eos_ids=eos_ids)
        print("\nWhat the critic reads for the first review:\n" + chat_text(tokenizer, first_messages))
    return LOADED["tokenizer"], LOADED["model"], LOADED["eos_ids"]


def save_batch(chunk, results, env):
    with open_for_append(CHECKPOINT) as handle:
        for row, result in zip(chunk.itertuples(), results):
            record = {"critic_run_key": CRITIC_RUN_KEY, "env_key": env_key(env), "env": env,
                      "batch_size": CRITIC_BATCH_SIZE, "task_key": row.task_key, "model": row.model,
                      "type": row.type, "id": row.id, "reviewed_utc": utc_now(), **result}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            DONE[row.task_key] = record


def review_model(name, group, env):
    """Review one model's answers that have no saved review yet, a batch at a time."""
    todo = group[~group["task_key"].isin(DONE)]
    started, new_tokens = time.time(), 0
    for start in tqdm(range(0, len(todo), CRITIC_BATCH_SIZE), desc=f"critic · {name}"):
        chunk = todo.iloc[start:start + CRITIC_BATCH_SIZE]
        message_lists = [critic_messages(row.model_prompt, row.response) for row in chunk.itertuples()]
        tokenizer, model, eos_ids = loaded_critic(message_lists[0])
        transformers.set_seed(SEED + start)  # unused when greedy
        try:
            results = generate_batch(tokenizer, model, eos_ids, message_lists)
        except torch.cuda.OutOfMemoryError:
            results = None  # handled below, once the failed batch's tensors have been released
        if results is None:
            free_gpu()
            raise RuntimeError(f"Out of GPU memory at critic batch size {CRITIC_BATCH_SIZE}. Set CRITIC_BATCH_SIZE = 16 "
                               "in section 2 and run all cells again; saved reviews are kept.")
        save_batch(chunk, results, env)
        new_tokens += sum(result["new_tokens"] for result in results)
    seconds = max(time.time() - started, 1e-9)
    return {"reviewed": len(todo), "minutes": round(seconds / 60, 2), "new_tokens_per_s": round(new_tokens / seconds),
            "finished_utc": utc_now()}

## 7. Review all 7,000 answers

In [11]:
LOG = {}
session_env = environment()
for name, group in tasks.groupby("model", sort=False):
    LOG[name] = review_model(name, group, session_env)
if LOADED:
    LOADED.clear()  # drop the critic, so the GPU memory is freed
    free_gpu()
print({name: entry["reviewed"] for name, entry in LOG.items()}, "reviews made in this session, by model")

critic · Llama-3.1-8B-Instruct:   0%|          | 0/32 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (9 outputs left out)

critic · Llama-SEA-LION-v3-8B-IT:   0%|          | 0/32 [00:00<?, ?it/s]

critic · Qwen-SEA-LION-v4-8B-VL:   0%|          | 0/32 [00:00<?, ?it/s]

critic · Llama3.1-typhoon2-8b-instruct:   0%|          | 0/32 [00:00<?, ?it/s]

{'Llama-3.1-8B-Instruct': 1000, 'Gemma-3-12b-it': 1000, 'Qwen3-VL-8B-Instruct': 1000, 'GLM-4-9b-chat': 1000, 'Llama-SEA-LION-v3-8B-IT': 1000, 'Qwen-SEA-LION-v4-8B-VL': 1000, 'Llama3.1-typhoon2-8b-instruct': 1000} reviews made in this session, by model


## 8. Anish's verdicts and the final answers

In [12]:
def excel_safe(text):
    """Experiment 1's cleanup: Windows line breaks become plain newlines, control characters are dropped."""
    if not isinstance(text, str):
        return text
    return ILLEGAL_CHARACTERS_RE.sub("", text.replace("\r\n", "\n").replace("\r", "\n"))


def review_record(row):
    """One answer's review: the critic's reply, Anish's verdict and final answer, and how the answer changed."""
    saved = DONE[row.task_key]
    final, meta = anish_review(row.model_prompt, row.response, saved["raw"])
    valid = isinstance(final, str)  # his code passes on whatever "corrected_response" held, even a non-string
    final_text = excel_safe(final) if valid else row.response
    return {**{column: getattr(row, column) for column in VERSION_COLUMNS}, "model": row.model,
            "candidate_response": row.response, "candidate_new_tokens": row.new_tokens,
            "candidate_hit_cap": row.hit_cap, "critic_raw": saved["raw"], "critic_new_tokens": saved["new_tokens"],
            "critic_hit_cap": saved["hit_cap"], "verdict": str(meta.get("verdict")),
            "critique": None if meta.get("critic_critique") is None else str(meta.get("critic_critique")),
            "critic_error": meta.get("error"), "rewritten": bool(valid and final != row.response),
            "final_invalid": not valid, "final_edited_for_excel": bool(valid and final_text != final),
            "response": final_text}


def in_study_order(frame):
    """Rows by model (notebook order), question (Experiment 1's order), then wording (original first)."""
    ranks = {"_model": frame["model"].map(MODEL_ORDER), "_id": frame["id"].map(QUESTION_ORDER),
             "_type": frame["type"].map(TYPE_ORDER)}
    return (frame.assign(**ranks).sort_values(list(ranks), kind="stable")
            .drop(columns=list(ranks)).reset_index(drop=True))


VERDICTS = ["BALANCED", "SYCOPHANTIC_REWRITE_REQUIRED", "CRITIC_ERROR"]


def model_summary(frame):
    verdicts = frame["verdict"]
    return {"reviews": len(frame), **{verdict: int((verdicts == verdict).sum()) for verdict in VERDICTS},
            "other_verdict": int((~verdicts.isin(VERDICTS)).sum()), "rewritten": int(frame["rewritten"].sum()),
            "final_invalid": int(frame["final_invalid"].sum()), "critic_hit_cap": int(frame["critic_hit_cap"].sum()),
            "empty_final": int((frame["response"].str.strip() == "").sum())}


missing = [key for key in tasks["task_key"] if key not in DONE]
if missing:
    raise RuntimeError(f"{len(missing)} reviews are missing. Run the review cell in section 7 again.")
reviews = in_study_order(pd.DataFrame([review_record(row) for row in tasks.itertuples()]))
by_model = pd.DataFrame({name: model_summary(group) for name, group in reviews.groupby("model", sort=False)}).T
by_model.loc["All models"] = model_summary(reviews)
by_model = by_model.rename_axis("model").reset_index()
display(by_model)

,model,reviews,BALANCED,SYCOPHANTIC_REWRITE_REQUIRED,CRITIC_ERROR,other_verdict,rewritten,final_invalid,critic_hit_cap,empty_final
0,Llama-3.1-8B-Instruct,1000,455,190,355,0,645,0,0,0
1,Gemma-3-12b-it,1000,455,29,516,0,484,0,0,0
2,Qwen3-VL-8B-Instruct,1000,474,273,253,0,728,0,0,0
3,GLM-4-9b-chat,1000,281,559,160,0,838,0,0,0
4,Llama-SEA-LION-v3-8B-IT,1000,562,28,410,0,474,0,0,0
5,Qwen-SEA-LION-v4-8B-VL,1000,569,280,151,0,805,0,0,0
6,Llama3.1-typhoon2-8b-instruct,1000,620,271,109,0,826,0,0,0
7,All models,7000,3416,1630,1954,0,4800,0,0,0


## 9. Save

In [13]:
def save_workbook(path, sheets):
    """Write the sheets to Excel, keeping text that starts with "=" as text rather than a formula."""
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame.to_excel(writer, sheet_name=name, index=False)
            for row in writer.sheets[name].iter_rows(min_row=2):
                for cell in row:
                    if cell.data_type == "f":  # text starting with "=" would otherwise be saved as a formula
                        cell.data_type = "s"


FINAL_FIELDS = VERSION_COLUMNS + ["model", "response", "candidate_response", "rewritten", "verdict", "final_invalid",
                                  "candidate_new_tokens", "candidate_hit_cap", "critic_hit_cap"]
write_jsonl(OUTPUT["final"], reviews[FINAL_FIELDS].to_dict("records"))
write_jsonl(OUTPUT["results"], reviews.to_dict("records"))
save_workbook(OUTPUT["xlsx"], {"results": reviews, "by_model": by_model})
sessions = {record["env_key"]: record["env"] for record in DONE.values()}
run = {"critic": CRITIC, "critic_run_key": CRITIC_RUN_KEY, "prism_fingerprint": PRISM_FINGERPRINT,
       "critic_sampling": CRITIC_SAMPLING, "critic_batch_size": CRITIC_BATCH_SIZE, "seed": SEED,
       "environments": list(sessions.values()), "this_session": LOG,
       "saved_by": {"python": platform.python_version(), **environment()}, "saved_utc": utc_now(),
       "by_model": by_model.to_dict("records")}
OUTPUT["run"].write_text(json.dumps(run, indent=2, default=str), encoding="utf-8")

saved = read_jsonl(OUTPUT["final"])
if [(r["model"], r["type"], r["id"], r["response"]) for r in saved] != list(
        zip(reviews["model"], reviews["type"], reviews["id"], reviews["response"])):
    raise RuntimeError(f"{OUTPUT['final'].name} does not read back as written. Run this cell again.")
print(f"Saved and checked: {', '.join(path.name for path in OUTPUT.values())} in {CRITIC_DIR}")
print("\nDone. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units. Next: notebook 9, "
      "prism_09_judge.ipynb, on a CPU runtime.")

Saved and checked: final_answers.jsonl, critic_results.jsonl, critic_results.xlsx, critic_run.json in /content/drive/MyDrive/CS612_prism/2_critic

Done. Runtime -> Disconnect and delete runtime, so the A100 stops using compute units. Next: notebook 9, prism_09_judge.ipynb, on a CPU runtime.
